# AzureML → Fabric: Push Model Artifacts

This notebook runs on **AzureML compute**. It downloads a trained model from the AzureML registry
and uploads the artifacts to a Microsoft Fabric Lakehouse via OneLake.

### Flow Overview

```
AzureML Registry → Download artifacts → Upload to Fabric Lakehouse → (done here)
                                                                     ↓
                               Fabric notebook: register model + activate endpoint
```

### After this notebook
Open `fabric_register_and_serve.ipynb` **in the Fabric portal** to register the model
with a proper signature and activate the endpoint. See the "What's Next" section at the bottom.

### Prerequisites
1. Run `az login` in the AzureML compute terminal
2. A Fabric workspace with at least one Lakehouse


## Configuration
Replace values if required

In [ ]:
FABRIC_WORKSPACE_NAME = "cargoluxmlfabric"
MODEL_DISPLAY_NAME = "credit_defaults_model"
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

## Step 1: Authenticate to Fabric

In [ ]:
import time, json, os, requests
from azure.identity import AzureCliCredential

fabric_credential = AzureCliCredential()
token = fabric_credential.get_token(FABRIC_SCOPE).token
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
print("[OK] Authenticated to Fabric")

resp = requests.get(f"{FABRIC_API_BASE}/workspaces", headers=headers)
resp.raise_for_status()
ws_match = [w for w in resp.json().get("value", []) if w["displayName"] == FABRIC_WORKSPACE_NAME]
if not ws_match:
    raise ValueError(f"Workspace '{FABRIC_WORKSPACE_NAME}' not found.")
WORKSPACE_ID = ws_match[0]["id"]
print(f"[OK] Workspace: {FABRIC_WORKSPACE_NAME} ({WORKSPACE_ID})")

def get_fabric_headers():
    t = fabric_credential.get_token(FABRIC_SCOPE).token
    return {"Authorization": f"Bearer {t}", "Content-Type": "application/json"}

## Step 2: Download Model from AzureML

In [ ]:
import mlflow
import yaml
from pathlib import Path

print(f"Downloading '{MODEL_DISPLAY_NAME}' from AzureML registry...")
local_path = mlflow.artifacts.download_artifacts(f"models:/{MODEL_DISPLAY_NAME}/latest")
print(f"[OK] Downloaded to: {local_path}")
print(f"Files: {os.listdir(local_path)}")

# Read model metadata
with open(os.path.join(local_path, "MLmodel")) as f:
    mlmodel = yaml.safe_load(f)

print(f"Flavors: {list(mlmodel.get('flavors', {}).keys())}")
sig = mlmodel.get("signature", {})
if sig.get("inputs"):
    inputs_schema = json.loads(sig["inputs"])
    print(f"Signature: {len(inputs_schema)} input features")
else:
    print("[INFO] No signature in MLmodel file (will be inferred in Fabric)")

## Step 3: Upload Artifacts to Fabric Lakehouse

In [ ]:
LAKEHOUSE_MODEL_DIR = f"azureml_models/{MODEL_DISPLAY_NAME}"

# Find a Lakehouse
headers = get_fabric_headers()
resp = requests.get(
    f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/items",
    params={"type": "Lakehouse"},
    headers=headers,
)
resp.raise_for_status()
lakehouses = resp.json().get("value", [])
if not lakehouses:
    raise ValueError("No Lakehouse found. Create one in the Fabric workspace first.")
LAKEHOUSE_ID = lakehouses[0]["id"]
LAKEHOUSE_NAME = lakehouses[0]["displayName"]
print(f"[OK] Using Lakehouse: {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")

# Upload via OneLake DFS
ONELAKE_DFS = "https://onelake.dfs.fabric.microsoft.com"
onelake_token = fabric_credential.get_token("https://storage.azure.com/.default").token
onelake_headers = {"Authorization": f"Bearer {onelake_token}"}

print(f"Uploading to Lakehouse Files/{LAKEHOUSE_MODEL_DIR}...")
artifact_root = Path(local_path)
for file_path in artifact_root.rglob("*"):
    if file_path.is_file():
        rel = file_path.relative_to(artifact_root).as_posix()
        onelake_file = f"{WORKSPACE_ID}/{LAKEHOUSE_ID}/Files/{LAKEHOUSE_MODEL_DIR}/{rel}"
        r = requests.put(f"{ONELAKE_DFS}/{onelake_file}?resource=file", headers=onelake_headers)
        data = file_path.read_bytes()
        requests.patch(
            f"{ONELAKE_DFS}/{onelake_file}?action=append&position=0",
            headers={**onelake_headers, "Content-Type": "application/octet-stream"}, data=data,
        )
        requests.patch(f"{ONELAKE_DFS}/{onelake_file}?action=flush&position={len(data)}", headers=onelake_headers)
        print(f"   [OK] {rel}")

print("[OK] All artifacts uploaded to Lakehouse")

abfs_model_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Files/{LAKEHOUSE_MODEL_DIR}"
print(f"\n⭐ ABFS path (copy this into the Fabric notebook config):\n   {abfs_model_path}")

## What's Next: Register & Serve in Fabric

The model artifacts are now in your Fabric Lakehouse. To register the model and create an endpoint:

### Option A: Manual (Recommended)
1. Go to the **Fabric portal** → your workspace
2. Upload or open the **`fabric_register_and_serve.ipynb`** notebook
3. **Attach your Lakehouse** to the notebook
4. Update `MODEL_PATH` with the ABFS path printed above
5. Run all cells — it registers the model with a proper signature and activates the endpoint

### Option B: Automated (API-triggered)
The cells below push the Fabric notebook via REST API and trigger it automatically.
This is useful for CI/CD but harder to debug. If endpoint activation fails, try Option A.


## (Optional) Automated: Push & Trigger Fabric Notebook

In [ ]:
import base64

def poll_lro(location_url, description="operation", max_attempts=60):
    for attempt in range(max_attempts):
        time.sleep(30)
        poll_resp = requests.get(location_url, headers=get_fabric_headers())
        poll_resp.raise_for_status()
        data = poll_resp.json()
        status = data.get("status", "Unknown")
        print(f"   [{attempt+1}] {description}: {status}")
        if status in ("Succeeded", "Completed"):
            return data
        if status == "Failed":
            raise RuntimeError(f"{description} failed: {json.dumps(data, indent=2)}")
    raise TimeoutError(f"{description} did not complete in {max_attempts} polls.")

# Read the Fabric notebook from this repo and inject config
with open("fabric_register_and_serve.ipynb", "r") as f:
    fabric_nb = json.load(f)

for cell in fabric_nb["cells"]:
    if cell.get("metadata", {}).get("tags") == ["parameters"]:
        cell["source"] = [
            f'MODEL_NAME = "{MODEL_DISPLAY_NAME}"\n',
            f'MODEL_PATH = "{abfs_model_path}"',
        ]
        break

nb_b64 = base64.b64encode(json.dumps(fabric_nb).encode()).decode()
print(f"[OK] Fabric notebook prepared")

# Find or create in Fabric
FABRIC_NB_NAME = "fabric_register_and_serve"
headers = get_fabric_headers()
resp = requests.get(
    f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/items",
    params={"type": "Notebook"}, headers=headers,
)
resp.raise_for_status()
nb_match = [n for n in resp.json().get("value", []) if n["displayName"] == FABRIC_NB_NAME]

if nb_match:
    FABRIC_NB_ID = nb_match[0]["id"]
    print(f"Updating notebook: {FABRIC_NB_NAME} ({FABRIC_NB_ID})")
    resp = requests.post(
        f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/notebooks/{FABRIC_NB_ID}/updateDefinition",
        headers=get_fabric_headers(),
        json={"definition": {"format": "ipynb", "parts": [{"path": "notebook-content.ipynb", "payload": nb_b64, "payloadType": "InlineBase64"}]}},
    )
    if resp.status_code == 202:
        loc = resp.headers.get("Location")
        if loc: poll_lro(loc, "Notebook update")
else:
    print(f"Creating notebook '{FABRIC_NB_NAME}'...")
    resp = requests.post(
        f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/notebooks",
        headers=get_fabric_headers(),
        json={"displayName": FABRIC_NB_NAME, "definition": {"format": "ipynb", "parts": [{"path": "notebook-content.ipynb", "payload": nb_b64, "payloadType": "InlineBase64"}]}},
    )
    if resp.status_code == 202:
        loc = resp.headers.get("Location")
        if loc: poll_lro(loc, "Notebook creation")
        resp2 = requests.get(f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/items", params={"type": "Notebook"}, headers=get_fabric_headers())
        resp2.raise_for_status()
        nb_match = [n for n in resp2.json().get("value", []) if n["displayName"] == FABRIC_NB_NAME]
    elif resp.status_code == 201:
        nb_match = [resp.json()]
    else:
        resp.raise_for_status()
    FABRIC_NB_ID = nb_match[0]["id"]
    print(f"[OK] Created: {FABRIC_NB_ID}")

print(f"\n[OK] Notebook ready in Fabric. Run next cell to trigger it.")

At this stage, you will have a notebook called `fabric_register_and_serve.ipynb` in your Fabric workspace. The lakehouse is not yet attached to the notebook, so make sure to do that manually on Fabric.

## **Below code does not work through API due to limitations, trigger the notebook "fabric_register_and_serve.ipynb manually" in Fabric!**

### (Optional) Trigger Notebook & Activate Endpoint

In [ ]:
# # Trigger the notebook in Fabric
# print("Triggering model registration...")
# run_resp = requests.post(
#     f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/items/{FABRIC_NB_ID}/jobs/RunNotebook/instances",
#     headers=get_fabric_headers(),
#     json={
#         "executionData": {
#             "configuration": {
#                 "defaultLakehouse": {"name": LAKEHOUSE_NAME, "id": LAKEHOUSE_ID, "workspaceId": WORKSPACE_ID},
#             },
#             "parameters": {
#                 "MODEL_NAME": {"value": MODEL_DISPLAY_NAME, "type": "string"},
#                 "MODEL_PATH": {"value": abfs_model_path, "type": "string"},
#             },
#         }
#     },
# )

# if run_resp.status_code == 202:
#     job_location = run_resp.headers.get("Location")
#     if job_location:
#         poll_lro(job_location, "Model registration")
#     print("[OK] Model registered in Fabric!")
# elif run_resp.status_code == 200:
#     print("[OK] Model registered in Fabric!")
# else:
#     print(f"Notebook trigger returned {run_resp.status_code}: {run_resp.text}")
#     run_resp.raise_for_status()

# # Look up model and activate endpoint
# time.sleep(10)
# resp = requests.get(
#     f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/items",
#     params={"type": "MLModel"}, headers=get_fabric_headers(),
# )
# resp.raise_for_status()
# model_match = [m for m in resp.json().get("value", []) if m["displayName"] == MODEL_DISPLAY_NAME]
# if not model_match:
#     raise ValueError(f"Model '{MODEL_DISPLAY_NAME}' not found after registration.")
# FABRIC_MODEL_ID = model_match[0]["id"]

# resp = requests.get(
#     f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/mlModels/{FABRIC_MODEL_ID}/versions",
#     headers=get_fabric_headers(),
# )
# MODEL_VERSION = "1"
# if resp.ok:
#     versions = resp.json().get("value", [])
#     if versions:
#         MODEL_VERSION = str(max(int(v.get("version", v.get("name", "1"))) for v in versions))
# print(f"[OK] Model: {MODEL_DISPLAY_NAME} v{MODEL_VERSION} ({FABRIC_MODEL_ID})")

# # Activate
# print(f"\nActivating endpoint for v{MODEL_VERSION}...")
# activate_url = (
#     f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}"
#     f"/mlmodels/{FABRIC_MODEL_ID}/endpoint/versions/{MODEL_VERSION}/activate"
# )
# resp = requests.post(activate_url, headers=get_fabric_headers())
# if resp.status_code == 200:
#     print("[OK] Endpoint activated!")
# elif resp.status_code == 202:
#     location = resp.headers.get("Location")
#     if not location:
#         raise RuntimeError("Got 202 but no Location header.")
#     poll_lro(location, "Endpoint activation")
#     print("[OK] Endpoint is now ACTIVE!")
# else:
#     resp.raise_for_status()